# Suite GEN — Sampling and generators

Samples are specified exactly on the random source's 32-bit words, so both implementations draw the same values from the
same seed. `Generate` streams new objects drawn from a distribution: each step chooses a case by weight, and builds what
the case's predicate requires by equality.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Distributions as D, Generators as G, Predicates as P, Queries as Q, Sampling as M, Validators as V } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Proxies, Schemas as S, Stores } from "@mbse/schemas/Framework";
import { Contact, Directory, assert, book, equal, listed, raises, same } from "./support.js";

/** A random source of the given words, for exact cases. */
class Words implements Stores.Random {
  readonly words: bigint[];

  constructor(...words: bigint[]) {
    this.words = words;
  }

  next_u32(): bigint {
    return this.words.shift() as bigint;
  }

  split(_key: string): Stores.Random {
    return this;
  }
}

function take<T>(items: Iterator<T>, n: number): T[] {
  return Array.from({ length: n }, () => items.next().value as T);
}

const APerson = { person: Contact };
const person = E.variable("person");
const HasName = new P.OfPredicate.Builder().name("HasName").symbols(APerson).parameters((p) => p.name("name"))
  .requires(person.name.eq(E.variable("name"))).create();
const named = (weight: number, name: string) => (a: D.Arm.Builder) => a.weight(weight).requires(HasName.call(person, name));
const Names = new P.OfPredicate.Builder().name("Names").symbols(APerson).requires(new D.Choices.Builder().arms(
  named(10, "alice"), named(5, "ben"), named(15, "chermon"), named(7, "davi")).decreasing().create()).create();
const MAX = (1n << 32n) - 1n;

## GEN-01 · Samples are specified on the source's words

In [ ]:
{
  let r: Stores.Random = new Stores.PCG32(42n, 54n);
  assert(same([0, 1, 2].map(() => M.uniform(r)), [0.6303102186438938, 0.7270080560068604, 0.7486033647998483]));
  r = new Stores.PCG32(42n, 54n);
  assert(same(take({ next: () => ({ value: M.below(r, 6n), done: false }) }, 8), [3n, 3n, 2n, 1n, 1n, 4n, 5n, 3n]));
  r = new Stores.PCG32(42n, 54n);
  assert(same([0, 1, 2].map(() => M.below(r, 2n ** 40n)), [788047328265n, 208370070163n, 325543878766n])); // two words each
  r = new Stores.PCG32(42n, 54n);
  assert(same(Array.from({ length: 10 }, () => M.weighted(r, [10, 5, 15, 7])), [2, 2, 2, 2, 3, 0, 3, 0, 2, 2]));
  assert(M.uniform(new Words(0n, 0n)) === 0 && M.uniform(new Words(MAX, MAX)) === 1 - 2 ** -53); // the ends
  assert(M.below(new Words(MAX, 5n), 3n) === 2n); // 2**32 - 1 is past the last multiple of 3: rejected, then 5 % 3
  assert(M.below(new Words(7n), 1n) === 0n && M.weighted(new Words(0n, 0n), [0, 2, 0]) === 1);
  assert(M.weighted(new Words(MAX, MAX), [1, 2, 0]) === 1); // the last positive weight, at the top
  raises(Errors.ValueError, () => M.below(new Words(), 0n), "below needs a positive int, got 0");
  raises(Errors.ValueError, () => M.weighted(new Words(), [1, -1]), "a weight must be a finite float, not negative, got -1.0");
  raises(Errors.ValueError, () => M.weighted(new Words(), [1, Infinity]), "a weight must be a finite float, not negative, got inf");
  raises(Errors.ValueError, () => M.weighted(new Words(), [0, 0]), "no weight is positive");
}

## GEN-02 · Generate chooses arms by weight and sets what their equalities require

In [ ]:
{
  const store = book();
  const generated = G.Generate(store, Names, new Stores.PCG32(42n));
  const first = take(generated, 12).map((m: any) => m.person.name);
  assert(same(first, ["chermon", "alice", "chermon", "ben", "davi", "davi", "chermon", "alice", "alice", "ben", "alice", "chermon"]));
  assert(same(take(G.Generate(book(), Names, new Stores.PCG32(42n)), 12).map((m: any) => m.person.name), first)); // the same seed
  assert(!same(take(G.Generate(book(), Names, new Stores.PCG32(7n)), 12).map((m: any) => m.person.name), first));
  assert(!same(take(G.Generate(book(), Names, new Stores.PCG32(42n).split("other")), 12).map((m: any) => m.person.name), first));
  const names = new Map<string, number>();
  for (const m of take(generated, 3700) as any[]) names.set(m.person.name, (names.get(m.person.name) ?? 0) + 1);
  for (const [n, w] of [["alice", 10], ["ben", 5], ["chermon", 15], ["davi", 7]] as const) {
    assert(Math.abs((names.get(n) as number) - 100 * w) < 100, n);
  }
  assert(generated[Symbol.iterator]() === generated && generated.steps === 3712 && generated.rejected === 0);
  const Fixed = new P.OfPredicate.Builder().symbols(APerson).requires(E.literal(30n).eq(person.age).and_(HasName.call(person, "cy")))
    .requires(person.name.eq(person.name), E.literal(1n).eq(1n)).create(); // either side of ==, through and and applications
  const made = (G.Generate(book(), Fixed, new Stores.PCG32(1n)).next().value as any).person;
  assert(made.name === "cy" && made.age === 30n); // the other equalities set nothing
}

## GEN-03 · A predicate that cannot be generated is refused

In [ ]:
{
  const Grown = new P.OfPredicate.Builder().symbols(APerson).requires(person.age.ge(18n)).create();
  raises(Errors.ValueError, () => G.Generate(book(), Grown, new Stores.PCG32(42n)).next(),
    "the predicate cannot be generated from its equalities and choices: what they build does not satisfy it");
  const Shadowed = new P.OfPredicate.Builder().symbols(APerson).requires(new D.Choices.Builder().arms(
    (a) => a.weight(1e-9).requires(person.has("name")), named(1, "alice")).decreasing().create()).create();
  raises(Errors.ValueError, () => G.Generate(book(), Shadowed, new Stores.PCG32(42n)).next(),
    "the predicate cannot be generated from its equalities and choices"); // alice falls under the first arm, not the one chosen
  raises(Errors.ValueError, () => G.Generate(book(), new P.OfPredicate.Builder().symbols(APerson).requires(
    new D.Choices.Builder().create()).create(), new Stores.PCG32(42n)), "the predicate cannot be drawn from: rule: a choices needs an arm");
  raises(Errors.ValueError, () => G.Generate(book(), HasName, new Stores.PCG32(42n)),
    "the predicate cannot be drawn from: a predicate with parameters is drawn from where it is applied");
  const Wrong = new P.OfPredicate.Builder().symbols(APerson).requires(
    new D.Normal.Builder().symbol("v").mean(0n).deviation(1n).requires(person.age.eq(E.variable("v"))).create()).create();
  raises(Errors.ValueError, () => G.Generate(book(), Wrong, new Stores.PCG32(42n)),
    "the predicate cannot be drawn from: person.age is int, but its distribution gives float");
}

## GEN-04 · Generated data is ordinary data: listed, validated, queried and written

In [ ]:
{
  const store = book();
  const people = take(G.Generate(store, Names, new Stores.PCG32(42n)), 6).map((m: any) => m.person);
  assert([...store.extent("Contact")].length === 0); // transient until linked to the store's data
  listed(store, ...people);
  assert([...store.extent("Contact")].length === 6);
  const NamedChermon = new P.OfPredicate.Builder().name("NamedChermon").symbols({ the: Contact })
    .requires(HasName.call(E.variable("the"), "chermon")).create();
  const chermons = [...new Q.Scan(store).select(NamedChermon)].map((m: any) => m.the);
  assert(same(chermons, people.filter((p: any) => p.name === "chermon")) && chermons.length > 0);
  const problems = V.Validate(store, [NamedChermon]).Reachable(Directory, store.singleton("book.Directory"));
  assert(problems.length === 6 - chermons.length && problems.every((p) => p.endsWith("'NamedChermon' does not hold")));
  assert(SchemaJSON.ToJSON(store).Reachable(Directory, store.singleton("book.Directory")).includes('"name": "ben"'));
}

## GEN-05 · Values are sampled exactly from the source's words

In [ ]:
{
  const r = new Stores.PCG32(42n, 54n);
  assert(same([0, 1].map(() => M.between(r, -1.0, 1.0)), [0.2606204372877876, 0.4540161120137207]));
  assert(same([0, 1].map(() => M.normal(r, 10.0, 2.0)), [11.674255764143341, 10.036009018664974]));
  assert(same([0, 1, 2, 3].map(() => M.poisson(r, 2.5)), [5n, 1n, 3n, 4n]) && M.poisson(r, 1234.5) === 1233n); // over 500: a sum
  assert(same([0, 1, 2, 3].map(() => M.geometric(r, 0.25)), [0n, 1n, 2n, 1n]) && M.geometric(new Words(), 1.0) === 0n);
  assert(same([1.0, 2.0, 0.5, 10.0, 1e-300, 5e-324].map((x) => M.log(x)),
    [0.0, 0.6931471805599453, -0.6931471805599453, 2.302585092994046, -690.7755278982137, -744.4400719213812]));
  assert(same([0.0, 1.0, -1.0, 0.1, 700.0, -740.0].map((x) => M.exp(x)),
    [1.0, 2.7182818284590455, 0.36787944117144233, 1.1051709180756477, 1.0142320547350045e+304, 4.2e-322]));
  assert(Number.isNaN(M.log(-1.0)) && Number.isNaN(M.log(NaN)) && M.log(0.0) === -Infinity && M.log(Infinity) === Infinity);
  assert(Number.isNaN(M.exp(NaN)) && M.exp(710.0) === Infinity && M.exp(-746.0) === 0);
  const [HALF, THREE_QUARTERS] = [2n ** 31n, 3n * 2n ** 30n];
  assert(M.normal(new Words(0n, 0n, 0n, 0n, HALF, 0n, HALF, 0n, THREE_QUARTERS, 0n, THREE_QUARTERS, 0n), 0.0, 1.0)
    === 0.5 * Math.sqrt(-2 * M.log(0.5) / 0.5)); // s = 2, then s = 0, are drawn again
  assert(M.poisson(new Words(MAX, MAX), 500.0) === 1583n); // u near 1: the sum stops when its terms underflow
  const cases: [() => unknown, string][] = [
    [() => M.between(new Words(), 1.0, 0.0), "a uniform's low must not exceed its high, got 1.0 and 0.0"],
    [() => M.between(new Words(), Infinity, 1.0), "a uniform's low must be finite, got inf"],
    [() => M.normal(new Words(), 0.0, -1.0), "a normal's deviation must not be negative, got -1.0"],
    [() => M.normal(new Words(), NaN, 1.0), "a normal's mean must be finite, got nan"],
    [() => M.poisson(new Words(), -1.0), "a Poisson's rate must not be negative, got -1.0"],
    [() => M.poisson(new Words(), Infinity), "a Poisson's rate must be finite, got inf"],
    [() => M.geometric(new Words(), 0.0), "a geometric's probability must be in (0, 1], got 0.0"],
  ];
  for (const [call, message] of cases) raises(Errors.ValueError, call, message);
}

## GEN-06 · Distributions draw what the equalities leave open, redrawn until the predicate holds

In [ ]:
{
  const aged = (distribution: any) => distribution.symbol("age").requires((age: any) => person.age.eq(age)).create();
  const Ages = new P.OfPredicate.Builder().symbols(APerson).requires(new D.Choices.Builder().arms(
    (a) => a.weight(3).requires(HasName.call(person, "senior"), aged(new D.Normal.Builder().mean(70n).deviation(8n).rounded()),
      person.age.ge(65n)),
    (a) => a.weight(7).requires(HasName.call(person, "adult"), aged(new D.Uniform.Builder().low(18n).high(64n)))).decreasing().create(),
  ).create();
  const generation = G.Generate(book(), Ages, new Stores.PCG32(42n));
  const people = take(generation, 1000).map((m: any) => m.person);
  const seniors = people.filter((p: any) => p.name === "senior").map((p: any) => p.age as bigint);
  const adults = people.filter((p: any) => p.name === "adult").map((p: any) => p.age as bigint);
  assert(Math.abs(seniors.length - 300) < 60 && seniors.every((v) => v >= 65n) && adults.every((v) => v >= 18n && v <= 64n));
  assert(generation.steps === 1000 && generation.rejected > 0); // seniors under 65, redrawn
  const Item = new S.OfObject.Builder().name("Item").ref().properties(
    (p) => p.name("a").of((t) => t.as_native(BigInt)), (p) => p.name("b").of((t) => t.as_native(BigInt)),
    (p) => p.name("c").of((t) => t.as_native(Number)), (p) => p.name("d").of((t) => t.as_native(Number))).create();
  const items: any = new Proxies.OfStore();
  items.register(Item);
  const [item, a] = [E.variable("item"), E.variable("a")];
  const Pairs = new P.OfPredicate.Builder().symbols({ item: Item }).requires(
    new D.Uniform.Builder().symbol("a").low(1n).high(6n).requires(item.a.eq(a), new D.Uniform.Builder().symbol("b").low(a).high(a)
      .requires(item.b.eq(E.variable("b"))).create()).create(),
    new D.Uniform.Builder().symbol("c").low(0.0).high(1.0).requires(item.c.eq(E.variable("c"))).create(), // floats in [0, 1)
    new D.Normal.Builder().symbol("d").mean(0.5).deviation(0.0).requires(item.d.eq(E.variable("d"))).create()).create();
  const pairs = [0n, 1n, 2n, 3n, 4n, 5n].map((seed) => (G.Generate(items, Pairs, new Stores.PCG32(seed)).next().value as any).item);
  assert(pairs.every((i: any) => i.b === i.a && i.a >= 1n && i.a <= 6n && i.c >= 0 && i.c < 1 && i.d === 0.5)
    && new Set(pairs.map((i: any) => i.a)).size > 1); // b reads a, as drawn
  const Fixed = new P.OfPredicate.Builder().symbols({ item: Item }).requires(item.a.eq(4n), item.b.eq(7n)).requires(
    new D.Uniform.Builder().symbol("a").low(1n).high(6n).requires(item.a.eq(a)).create()).create();
  const fixed = (G.Generate(items, Fixed, new Stores.PCG32(1n)).next().value as any).item;
  assert(fixed.a === 4n && fixed.b === 7n); // what an equality sets first is not drawn
  const Between = new P.OfPredicate.Builder().name("Between").symbols({ item: Item }).parameters((p) => p.name("low")).requires(
    new D.Uniform.Builder().symbol("a").low(E.variable("low")).high(E.variable("low")).requires(item.a.eq(a)).create()).create();
  const Applied = new P.OfPredicate.Builder().symbols({ item: Item }).requires(Between.call(item, E.literal(3n).add(2n))).create();
  assert((G.Generate(items, Applied, new Stores.PCG32(1n)).next().value as any).item.a === 5n); // an argument, evaluated for the walk
  const Never = new P.OfPredicate.Builder().symbols({ item: Item }).requires(
    new D.Uniform.Builder().symbol("a").low(1n).high(6n).requires(item.a.eq(a)).create(), item.a.gt(6n)).create();
  const never = G.Generate(items, Never, new Stores.PCG32(1n));
  raises(Errors.ValueError, () => never.next(), "the predicate cannot be generated: none of its 100 attempts satisfies it");
  assert(never.rejected === G.ATTEMPTS && never.steps === 0);
  const Reversed = new P.OfPredicate.Builder().symbols({ item: Item }).requires(
    new D.Uniform.Builder().symbol("a").low(6n).high(1n).requires(item.a.eq(a)).create()).create();
  raises(Errors.ValueError, () => G.Generate(items, Reversed, new Stores.PCG32(1n)).next(), "a uniform's low must not exceed its high, got 6 and 1");
  const Text = new P.OfPredicate.Builder().symbols({ item: Item }).requires(
    new D.Poisson.Builder().symbol("a").rate("x").requires(item.a.eq(a)).create()).create();
  raises(TypeError, () => G.Generate(items, Text, new Stores.PCG32(1n)).next(), "a Poisson's rate must be a number, got str");
  const Picked = new P.OfPredicate.Builder().symbols({ item: Item }).requires(
    new D.Categorical.Builder().symbol("a").option(1, 2n).option(3, 5n).requires(item.a.eq(a)).create(),
    new D.Geometric.Builder().symbol("b").probability(1.0).requires(item.b.eq(E.variable("b"))).create()).create();
  const picked = [0n, 1n, 2n, 3n, 4n, 5n, 6n, 7n].map((seed) => (G.Generate(items, Picked, new Stores.PCG32(seed)).next().value as any).item);
  assert(same([...new Set(picked.map((i: any) => i.a))].sort(), [2n, 5n]) && same([...new Set(picked.map((i: any) => i.b))], [0n]));
}